
# Dayton WRF — Odor Mitigation Analysis Notebook

This notebook unifies sludge flow data, continuous NH₃/H₂S monitors, and operational notes to analyze the effects of **HCl** and **FeCl₃** dosing on odors from anaerobically digested biosolids.

**Built for:** Phil Bennington's candidacy research description and full‑scale Dayton WRF operations (2025).  
**Author (auto‑generated):** ChatGPT companion notebook

---

## What this notebook does
1. **Load & standardize**: Sludge flow (east/west), NH₃ monitor logs, H₂S monitor logs.
2. **Align by time**: Parse timestamps, localize to US/Eastern, resample to hourly/daily.
3. **Compute dosage** (mg/L) from **lbs/day (solution)** and plant flow (MGD or gallons), including 100%‑strength conversion.
4. **Add operational windows**: HCl OFF, FeCl₃ OFF, meters removed dates.
5. **Visualize** time series with shaded windows; dual‑axis NH₃/H₂S; dosage trends.
6. **Analyze** relationships: before/after means, correlations, and scatter of gas vs dosage.
7. **Export** clean figures.

> **Cite:** “Investigations on Odor Elimination with Hydrochloric Acid and Ferric Chloride Application to Biosolids Matrix — Pre‑Proposal (Feb 28, 2025)” (Dayton WRF full‑scale context, feed rates, and mechanisms).


# Imports

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
import pytz
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import os

pio.renderers.default = "vscode"

pd.options.display.float_format = lambda v: f"{v:,.3f}"


## 1) Configure file paths (edit these)
Fill these with your actual files. Supported formats: CSV/XLSX exported from your sources.



In [18]:
# ============================================
# PATHS & SETTINGS
# ============================================
data_path = Path("/Users/mutsamungoshi/Documents/Wastewater Treatment/data")
all_files = list(data_path.glob("*.xlsx"))

combined = []
diagnostics = []
skipped_files = []
column_coverage = []
bad_rows = []

EXPECTED_COLS = [
    "Time",
    "West Sludge Out (GPM)",
    "West Sludge Flow Today (Gal)",
    "East Sludge Out (GPM)",
    "East Sludge Flow Today (Gal)",
    "West Sludge Tank Level (FT)",
    "GBT Sludge Feed Pump 1 (GPM)",
    "GBT Sludge Feed Pump 2 (GPM)"
]

# ============================================
# SAFE TIME PARSER
# ============================================
def safe_parse(s):
    try:
        return datetime.strptime(str(s).strip(), "%m/%d/%y %I:%M:%S %p")
    except:
        return pd.NaT

# ============================================
# FILE LOADER (FIXED)
# ============================================
def load_one_file(file):
    try:
        # Try reading normally
        df = pd.read_excel(file, skiprows=0)

        # If first column is not 'Time', try skipping the first row
        if df.columns[0] != "Time":
            df = pd.read_excel(file, skiprows=1)

        # Keep only expected columns
        df = df.iloc[:, :len(EXPECTED_COLS)]
        df.columns = EXPECTED_COLS

        # Drop repeated headers INSIDE the sheet
        df = df[df["Time"] != "Time"].copy()

        # Drop blank rows
        df.dropna(how="all", inplace=True)

        # Parse datetimes
        df["Datetime"] = df["Time"].apply(safe_parse)

        # Capture rows that failed datetime parsing
        bad_dt = df[df["Datetime"].isna()].copy()
        if not bad_dt.empty:
            bad_dt["Source_File"] = file.name
            bad_rows.append(bad_dt)

        # Remove unparseable rows
        df = df[df["Datetime"].notna()].copy()

        # Fail if no timestamps survive
        if df.empty:
            raise ValueError("No valid timestamp rows.")

        df["Source_File"] = file.name
        return df

    except Exception as e:
        skipped_files.append({"file": file.name, "reason": str(e)})
        return None

# ============================================
# MAIN LOOP
# ============================================
for file in all_files:
    # Skip non-data files
    if "Tile" in file.name or "~$" in file.name:
        skipped_files.append({"file": file.name, "reason": "Skipped non-data file"})
        continue

    print(f"📄 Processing: {file.name}")
    df = load_one_file(file)

    if df is None or df.empty:
        print(f"❌ Empty or failed: {file.name}")
        continue

    # Track column coverage
    column_coverage.append({
        "File": file.name,
        **{col: col in df.columns for col in EXPECTED_COLS}
    })

    # Diagnostics
    diagnostics.append({
        "File": file.name,
        "Rows": len(df),
        "Start": df["Datetime"].min(),
        "End": df["Datetime"].max(),
        "Columns Present": df.shape[1]
    })

    combined.append(df)
    print(f"✅ Loaded {len(df)} rows, {df.shape[1]} cols")

# ============================================
# EXPORTS
# ============================================
if combined:
    final_df = pd.concat(combined, ignore_index=True)
    final_df.to_csv(data_path / "combined_sludge_data.csv", index=False)
    print("💾 Exported combined data")

if diagnostics:
    pd.DataFrame(diagnostics).to_csv(data_path / "file_diagnostics_summary.csv", index=False)

if skipped_files:
    pd.DataFrame(skipped_files).to_csv(data_path / "skipped_files_log.csv", index=False)

if column_coverage:
    pd.DataFrame(column_coverage).set_index("File").to_csv(data_path / "column_coverage.csv")
else:
    print("⚠️ No non-empty DataFrames for column coverage.")

if bad_rows:
    pd.concat(bad_rows).to_csv(data_path / "bad_datetime_rows.csv", index=False)
    print(f"⚠️ Exported {sum(len(df) for df in bad_rows)} bad datetime rows to CSV")

# ============================================================
# 2️⃣ Set your input paths (all under /data)
# ============================================================

data_dir = Path("/Users/mutsamungoshi/Documents/Wastewater Treatment/data")

# Sludge flow, tank levels, pumps (combined .csv)
SLUDGE_PATH = data_dir / "combined_sludge_data.csv"

# Acrulog sensor exports
NH3_PATH = data_dir / "NH3-211205141_[9-12-2025, 7-22-54 AM --- 10-8-2025, 8-07-54 AM].csv"
H2S_PATH = data_dir / "H2S-240708934_10-08-25_[9-12-2025, 7-23-08 AM --- 10-8-2025, 8-08-08 AM].csv"

# Optional: chemical feed log (if available)
FEED_PATH = None  # e.g., data_dir / "daily_chemical_feed_log.csv"

# === Verification summary ===
def print_path_summary():
    print("\n📂 Path summary:")
    for label, path in {
        "SLUDGE_PATH": SLUDGE_PATH,
        "NH3_PATH": NH3_PATH,
        "H2S_PATH": H2S_PATH,
        "FEED_PATH": FEED_PATH,
    }.items():
        exists = path.exists() if path else False
        status = "✅ FOUND" if exists else "❌ MISSING"
        print(f"{label:12}: {str(path) if path else '—':70} {status}")

print_path_summary()


📄 Processing: Water Reclamation Dig SLG Out_09_20_2025 06_45_00 AM.xlsx
✅ Loaded 1440 rows, 10 cols
📄 Processing: Water Reclamation Dig SLG Out_10_05_2025 06_45_00 AM.xlsx
✅ Loaded 1440 rows, 10 cols
📄 Processing: Water Reclamation Dig SLG Out_08_26_2025 06_45_00 AM.xlsx
✅ Loaded 1440 rows, 10 cols
📄 Processing: Water Reclamation Dig SLG Out_09_28_2025 06_45_00 AM.xlsx
✅ Loaded 1440 rows, 10 cols
📄 Processing: Water Reclamation Dig SLG Out_08_30_2025 06_45_00 AM.xlsx
✅ Loaded 1440 rows, 10 cols
📄 Processing: Water Reclamation Dig SLG Out_09_12_2025 06_45_00 AM.xlsx
✅ Loaded 1440 rows, 10 cols
📄 Processing: Water Reclamation Dig SLG Out_09_04_2025 06_45_00 AM.xlsx
✅ Loaded 1440 rows, 10 cols
📄 Processing: Water Reclamation Dig SLG Out_09_18_2025 06_45_00 AM.xlsx
✅ Loaded 1440 rows, 10 cols
📄 Processing: Water Reclamation Dig SLG Out_10_07_2025 06_45_00 AM.xlsx
✅ Loaded 1440 rows, 10 cols
📄 Processing: Water Reclamation Dig SLG Out_09_22_2025 06_45_00 AM.xlsx
✅ Loaded 1440 rows, 10 cols



## 2) Load & standardize data
This will try to read CSV **or** Excel, normalize **Datetime** columns, and set up frames:
- `sludge` (minute-level or hourly)
- `nh3` (monitor series)
- `h2s` (monitor series)


In [3]:
import pandas as pd
import pytz
from pathlib import Path

# ============================================================
# Helper functions
# ============================================================
def load_table(path: Path, sheet_name=None):
    """Generic CSV/Excel loader with automatic file-type detection and header-skip detection."""
    if not path or not path.exists():
        print(f"⚠️ File not found: {path}")
        return pd.DataFrame()

    # Excel
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path, sheet_name=sheet_name)

    # CSV — try to auto-detect header line
    try:
        with open(path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        header_line = 0
        for i, line in enumerate(lines[:15]):  # look at first 15 lines
            if "ISO time" in line and "," in line:
                header_line = i
                break

        df = pd.read_csv(path, skiprows=header_line)
        return df

    except Exception as e:
        print(f"❌ Failed to read {path.name}: {e}")
        return pd.DataFrame()


def parse_time_columns(df, candidates=("ISO time", "Datetime", "Time Stamp", "Time")):
    """Find and convert an appropriate time column (Acrulog exports use ISO UTC time)."""
    for c in candidates:
        if c in df.columns:
            out = pd.to_datetime(df[c], errors="coerce", utc=True)
            if out.notna().sum() > 0:
                return out.rename("Datetime")
    return pd.to_datetime(df.index, errors="coerce", utc=True).rename("Datetime")

# ============================================================
# Timezone setup
# ============================================================
tz = pytz.timezone("US/Eastern")

# ============================================================
# Load Sludge Data
# ============================================================
sludge = load_table(SLUDGE_PATH)
if not sludge.empty:
    if "Datetime" not in sludge.columns:
        sludge["Datetime"] = parse_time_columns(sludge)
    else:
        sludge["Datetime"] = pd.to_datetime(sludge["Datetime"], errors="coerce", utc=True)
    sludge["Datetime"] = sludge["Datetime"].dt.tz_convert(tz)
    sludge = sludge.dropna(subset=["Datetime"]).sort_values("Datetime").reset_index(drop=True)

# ============================================================
# Load NH3 Data (Acrulog)
# ============================================================
nh3 = load_table(NH3_PATH)
if not nh3.empty:
    nh3["Datetime"] = parse_time_columns(nh3)
    nh3["Datetime"] = nh3["Datetime"].dt.tz_convert(tz)
    nh3.rename(
        columns={
            "NH3 (PPM)": "NH3_ppm",
            "Temperature (°F)": "Temp_F",
            "Humidity (%)": "Humidity_pct",
        },
        inplace=True,
    )
    keep_cols = [c for c in ["Datetime", "NH3_ppm", "Temp_F", "Humidity_pct"] if c in nh3.columns]
    nh3 = nh3[keep_cols].dropna(subset=["Datetime"]).sort_values("Datetime").reset_index(drop=True)

# ============================================================
# Load H2S Data (Acrulog)
# ============================================================
h2s = load_table(H2S_PATH)
if not h2s.empty:
    h2s["Datetime"] = parse_time_columns(h2s)
    h2s["Datetime"] = h2s["Datetime"].dt.tz_convert(tz)
    h2s.rename(
        columns={
            "H2S (PPM)": "H2S_ppm",
            "Temperature (°F)": "Temp_F",
            "Humidity (%)": "Humidity_pct",
        },
        inplace=True,
    )
    keep_cols = [c for c in ["Datetime", "H2S_ppm", "Temp_F", "Humidity_pct"] if c in h2s.columns]
    h2s = h2s[keep_cols].dropna(subset=["Datetime"]).sort_values("Datetime").reset_index(drop=True)

print("✅ Loaded shapes -> sludge:", sludge.shape, "nh3:", nh3.shape, "h2s:", h2s.shape)

✅ Loaded shapes -> sludge: (77760, 10) nh3: (12496, 4) h2s: (12496, 4)



## 3) Resample to hourly & daily; build the `daily` table


In [4]:
# ============================================================
# 🧪 DAILY AGGREGATION — BENNINGTON STUDY ALIGNED (Final Fix)
# ============================================================
from datetime import date
import pandas as pd

import pandas as pd

def coerce_datetime_eastern(df, datetime_col="Datetime"):
    """
    Convert a datetime column to timezone-aware US/Eastern datetimes.
    """
    # Parse the column
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col], errors="coerce", utc=True)
    # Convert to Eastern Time
    df[datetime_col] = df[datetime_col].dt.tz_convert("US/Eastern")
    return df

TZ = "US/Eastern"

# --- 1️⃣ Sludge daily ---
sludge_fixed = coerce_datetime_eastern(sludge)
sludge_fixed["Total Flow (Gal)"] = (
    pd.to_numeric(sludge_fixed["East Sludge Flow Today (Gal)"], errors="coerce").fillna(0)
    + pd.to_numeric(sludge_fixed["West Sludge Flow Today (Gal)"], errors="coerce").fillna(0)
)
sludge_fixed["LocalDate"] = sludge_fixed["Datetime"].dt.tz_convert(TZ).dt.date
daily_sludge = (
    sludge_fixed.groupby("LocalDate", as_index=False)["Total Flow (Gal)"]
    .max()
    .assign(**{"Flow (MGD)": lambda x: x["Total Flow (Gal)"] / 1_000_000})
)

# --- 2️⃣ NH3 daily ---
if not nh3.empty:
    nh3_fixed = coerce_datetime_eastern(nh3)
    nh3_fixed.rename(columns={"NH3 (PPM)": "NH3_ppm"}, inplace=True)
    nh3_fixed["LocalDate"] = nh3_fixed["Datetime"].dt.tz_convert(TZ).dt.date
    nh3_fixed["NH3_ppm"] = pd.to_numeric(nh3_fixed["NH3_ppm"], errors="coerce")
    nh3_daily = nh3_fixed.groupby("LocalDate", as_index=False)["NH3_ppm"].mean()
else:
    nh3_daily = pd.DataFrame(columns=["LocalDate", "NH3_ppm"])

# --- 3️⃣ H2S daily ---
if not h2s.empty:
    h2s_fixed = coerce_datetime_eastern(h2s)
    h2s_fixed.rename(columns={"H2S (PPM)": "H2S_ppm"}, inplace=True)
    h2s_fixed["LocalDate"] = h2s_fixed["Datetime"].dt.tz_convert(TZ).dt.date
    h2s_fixed["H2S_ppm"] = pd.to_numeric(h2s_fixed["H2S_ppm"], errors="coerce")
    h2s_daily = h2s_fixed.groupby("LocalDate", as_index=False)["H2S_ppm"].mean()
else:
    h2s_daily = pd.DataFrame(columns=["LocalDate", "H2S_ppm"])

# --- 4️⃣ Verify ranges ---
print("🧾 Sludge:", daily_sludge["LocalDate"].min(), "→", daily_sludge["LocalDate"].max())
if not nh3_daily.empty:
    print("NH3 :", nh3_daily["LocalDate"].min(), "→", nh3_daily["LocalDate"].max())
if not h2s_daily.empty:
    print("H2S :", h2s_daily["LocalDate"].min(), "→", h2s_daily["LocalDate"].max())

# --- 5️⃣ Merge ---
daily = (
    daily_sludge
    .merge(nh3_daily, on="LocalDate", how="left")
    .merge(h2s_daily, on="LocalDate", how="left")
)

# --- 6️⃣ Add Phase annotation ---
HCL_OFF_START = date(2025, 8, 27)
HCL_RESUME = date(2025, 10, 2)
def assign_phase(d):
    if pd.isna(d): return None
    if d < HCL_OFF_START:
        return "Pre-shutdown"
    elif d < HCL_RESUME:
        return "Chemical OFF (HCl & partial Ferric)"
    else:
        return "Post-resumption"

daily["Phase"] = daily["LocalDate"].apply(assign_phase)

# --- ✅ Final summary ---
print("\n✅ Final daily shape:", daily.shape)
print("🗓 Range:", daily["LocalDate"].min(), "→", daily["LocalDate"].max())
print("📊 Columns:", list(daily.columns))
display(daily.head(10))

🧾 Sludge: 2025-08-13 → 2025-10-07
NH3 : 2025-09-12 → 2025-10-08
H2S : 2025-09-12 → 2025-10-08

✅ Final daily shape: (56, 6)
🗓 Range: 2025-08-13 → 2025-10-07
📊 Columns: ['LocalDate', 'Total Flow (Gal)', 'Flow (MGD)', 'NH3_ppm', 'H2S_ppm', 'Phase']


,LocalDate,Total Flow (Gal),Flow (MGD),NH3_ppm,H2S_ppm,Phase
0,2025-08-13,"82,881.070",0.083,NaN,NaN,Pre-shutdown
1,2025-08-14,"348,404.470",0.348,NaN,NaN,Pre-shutdown
2,2025-08-15,"69,047.160",0.069,NaN,NaN,Pre-shutdown
3,2025-08-16,"362,825.240",0.363,NaN,NaN,Pre-shutdown
4,2025-08-17,"308,674.220",0.309,NaN,NaN,Pre-shutdown
5,2025-08-18,"447,245.680",0.447,NaN,NaN,Pre-shutdown
6,2025-08-19,"550,918.120",0.551,NaN,NaN,Pre-shutdown
7,2025-08-20,"262,219.260",0.262,NaN,NaN,Pre-shutdown
8,2025-08-21,"570,341.720",0.570,NaN,NaN,Pre-shutdown
9,2025-08-22,"524,366.200",0.524,NaN,NaN,Pre-shutdown



## 4) Operational windows (from plant notes)
- **HCl OFF**: 2025‑08‑27 00:00 → 2025‑10‑02 14:30 (delivery/restart)
- **FeCl₃ OFF**: 2025‑09‑09 08:00 → 2025‑09‑17 12:00
- **Meters removed**: 2025‑10‑08 08:00


In [5]:
import pytz
import pandas as pd

# ============================================================
# Define Time Markers (Bennington Field Notes, Eastern Time)
# ============================================================
def to_eastern(ts):
    """Convert string or timestamp to US/Eastern tz-aware datetime."""
    tz = pytz.timezone("US/Eastern")
    t = pd.to_datetime(ts)
    if t.tzinfo is None:
        return t.tz_localize(tz)
    return t.tz_convert(tz)

# --- Key operational events ---
HCL_OFF_START   = to_eastern("2025-08-27 00:00")
HCL_RESUME      = to_eastern("2025-10-02 14:30")
FERRIC_OFF      = to_eastern("2025-09-09 08:00")
FERRIC_ON       = to_eastern("2025-09-17 12:00")
SENSORS_REMOVED = to_eastern("2025-10-08 08:00")

# ============================================================
# Plotly Helper for Datetime Event Lines
# ============================================================
def add_dt_vline(fig, x_dt, text="Event", line_kwargs=None, ann_kwargs=None):
    """
    Adds a vertical line + annotation to a Plotly figure at a given datetime.
    
    Parameters
    ----------
    fig : go.Figure
        Plotly figure object.
    x_dt : datetime
        Target datetime (can be tz-aware).
    text : str
        Annotation label for the line.
    line_kwargs : dict
        Line style overrides.
    ann_kwargs : dict
        Annotation style overrides.
    """
    # Convert to timezone-naive for Plotly (avoid mixed tz errors)
    x_py = getattr(x_dt, "to_pydatetime", lambda: x_dt)()
    if getattr(x_py, "tzinfo", None) is not None:
        x_py = x_py.replace(tzinfo=None)
    
    if line_kwargs is None:
        line_kwargs = dict(color="gray", width=2, dash="dot")
    
    if ann_kwargs is None:
        ann_kwargs = dict(
            y=1.02, yref="paper", showarrow=False,
            font=dict(color="gray", size=12),
            xanchor="left", yanchor="bottom"
        )

    fig.add_shape(
        type="line", x0=x_py, x1=x_py, y0=0, y1=1,
        xref="x", yref="paper", line=line_kwargs
    )
    fig.add_annotation(x=x_py, text=text, **ann_kwargs)
    return fig


## 5) Dosage math (from operations/report)

- Reported feeds (solution): **HCl 32% = 6,230 lb/day**, **FeCl₃ 37.9% = 583 lb/day**  
- Convert to *100%‑strength* equivalents:  
  - `lb100_HCl = 6,230 × 0.32 = 1,993.6 lb/day`  
  - `lb100_FeCl3 = 583 × 0.379 ≈ 220.9 lb/day`  
- **Conversion**: `MGD × 8.34 × mg/L = lb 100% strength` → `mg/L = lb100 / (8.34 × MGD)`


In [6]:
# ============================================================
# Chemical Dosage Calculation — Phase-Aware (Bennington 2025)
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Constants (Bennington field study)
# ------------------------------------------------------------
LB_SOLUTION_HCL   = 6230.0   # lbs/day of 32% HCl solution
HCL_FRAC          = 0.32
LB_SOLUTION_FECL3 = 583.0    # lbs/day of 37.9% FeCl₃ solution
FECL3_FRAC        = 0.379

LB100_HCL   = LB_SOLUTION_HCL * HCL_FRAC     # 1,994 lbs/day pure HCl
LB100_FECL3 = LB_SOLUTION_FECL3 * FECL3_FRAC # 221 lbs/day pure FeCl₃

# ------------------------------------------------------------
# Helper: lbs → mg/L
# ------------------------------------------------------------
def dosage_mgL(lb100, mgd):
    """Convert daily lbs (100% basis) and flow (MGD) into mg/L."""
    try:
        return lb100 / (8.34 * mgd) if (mgd and mgd > 0) else np.nan
    except Exception:
        return np.nan

# ------------------------------------------------------------
# Field-based OFF/ON windows (Bennington log)
# ------------------------------------------------------------
HCL_OFF_START  = pd.Timestamp("2025-08-27")
HCL_RESUME     = pd.Timestamp("2025-10-02 14:30")
FECL3_OFF      = pd.Timestamp("2025-09-09 08:00")
FECL3_RESUME   = pd.Timestamp("2025-09-17 12:00")

def hcl_feed_lbs(date):
    """Return HCl lbs/day depending on OFF/ON status."""
    if HCL_OFF_START <= date < HCL_RESUME:
        return 0.0
    return LB100_HCL

def fecl3_feed_lbs(date):
    """Return FeCl₃ lbs/day depending on OFF/ON status."""
    if FECL3_OFF <= date < FECL3_RESUME:
        return 0.0
    return LB100_FECL3

# ------------------------------------------------------------
# Compute mg/L dosage using actual operational status
# ------------------------------------------------------------
if not daily.empty and "Flow (MGD)" in daily.columns:
    daily["HCl Dosage (mg/L)"] = daily.apply(
        lambda r: dosage_mgL(hcl_feed_lbs(pd.Timestamp(r["LocalDate"])), r["Flow (MGD)"]),
        axis=1,
    )
    daily["FeCl₃ Dosage (mg/L)"] = daily.apply(
        lambda r: dosage_mgL(fecl3_feed_lbs(pd.Timestamp(r["LocalDate"])), r["Flow (MGD)"]),
        axis=1,
    )
    print("✅ Computed field-aligned dosage columns (OFF periods masked).")
else:
    print("⚠️ Missing flow data — cannot compute mg/L dosage.")

# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------
cols = ["LocalDate", "Flow (MGD)", "HCl Dosage (mg/L)", "FeCl₃ Dosage (mg/L)", "Phase"]
display(daily[cols].head(15))

✅ Computed field-aligned dosage columns (OFF periods masked).


,LocalDate,Flow (MGD),HCl Dosage (mg/L),FeCl₃ Dosage (mg/L),Phase
0,2025-08-13,0.083,"2,884.142",319.659,Pre-shutdown
1,2025-08-14,0.348,686.101,76.043,Pre-shutdown
2,2025-08-15,0.069,"3,461.993",383.704,Pre-shutdown
3,2025-08-16,0.363,658.832,73.020,Pre-shutdown
4,2025-08-17,0.309,774.411,85.830,Pre-shutdown
5,2025-08-18,0.447,534.473,59.237,Pre-shutdown
6,2025-08-19,0.551,433.895,48.090,Pre-shutdown
7,2025-08-20,0.262,911.606,101.036,Pre-shutdown
8,2025-08-21,0.570,419.119,46.452,Pre-shutdown
9,2025-08-22,0.524,455.866,50.525,Pre-shutdown


In [15]:
import pandas as pd
import numpy as np
import pytz

# --- Constants ---
LB_SOLUTION_HCL   = 6230.0   # lbs/day of 32% HCl
HCL_FRAC          = 0.32
LB_SOLUTION_FECL3 = 583.0    # lbs/day of 37.9% FeCl₃
FECL3_FRAC        = 0.379

LB100_HCL   = LB_SOLUTION_HCL * HCL_FRAC     # Pure HCl
LB100_FECL3 = LB_SOLUTION_FECL3 * FECL3_FRAC # Pure FeCl₃

def dosage_mgL(lb100, mgd):
    """Convert daily lbs (100% basis) and flow (MGD) into mg/L."""
    try:
        return lb100 / (8.34 * mgd) if (mgd and mgd > 0) else np.nan
    except Exception:
        return np.nan

# --- Timezone ---
def to_eastern(ts):
    tz = pytz.timezone("US/Eastern")
    t = pd.to_datetime(ts)
    if t.tzinfo is None:
        return t.tz_localize(tz)
    return t.tz_convert(tz)

# --- OFF/ON windows (Eastern) ---
HCL_OFF_START_DT   = to_eastern("2025-08-27 00:00")
HCL_RESUME_DT      = to_eastern("2025-10-02 14:30")
FECL3_OFF_DT       = to_eastern("2025-09-09 08:00")
FECL3_RESUME_DT    = to_eastern("2025-09-17 12:00")

def hourly_hcl_lbs(ts):
    if HCL_OFF_START_DT <= ts < HCL_RESUME_DT:
        return 0.0
    return LB100_HCL

def hourly_fecl3_lbs(ts):
    if FECL3_OFF_DT <= ts < FECL3_RESUME_DT:
        return 0.0
    return LB100_FECL3

# --- Resample to hourly flow (assumes sludge_fixed is timezone-aware and has 'East Sludge Flow Today (Gal)', 'West Sludge Flow Today (Gal)' columns) ---
sludge_hourly = sludge_fixed.copy()
sludge_hourly["Total Flow (Gal)"] = (
    pd.to_numeric(sludge_hourly["East Sludge Flow Today (Gal)"], errors="coerce").fillna(0) +
    pd.to_numeric(sludge_hourly["West Sludge Flow Today (Gal)"], errors="coerce").fillna(0)
)
sludge_hourly = sludge_hourly.set_index("Datetime").sort_index()

hourly_flow = (
    sludge_hourly["Total Flow (Gal)"]
    .resample("1H")
    .max()  # Use .mean() if more appropriate
    .dropna()
    .to_frame()
    .rename(columns={"Total Flow (Gal)": "Flow (Gal)"})
)
hourly_flow["Flow (MGD)"] = hourly_flow["Flow (Gal)"] / 1_000_000
hourly_flow = hourly_flow.reset_index()

# --- Apply mg/L dosage calculations ---
hourly_flow["HCl Dosage (mg/L)"] = hourly_flow.apply(
    lambda row: dosage_mgL(hourly_hcl_lbs(row["Datetime"]), row["Flow (MGD)"]),
    axis=1
)

hourly_flow["FeCl₃ Dosage (mg/L)"] = hourly_flow.apply(
    lambda row: dosage_mgL(hourly_fecl3_lbs(row["Datetime"]), row["Flow (MGD)"]),
    axis=1
)

# --- Preview ---
cols = ["Datetime", "Flow (MGD)", "HCl Dosage (mg/L)", "FeCl₃ Dosage (mg/L)"]
display(hourly_flow[cols].head(15))

# --- Optional export ---
output_hourly_path = data_path / "hourly_chemical_dosage.csv"
hourly_flow.to_csv(output_hourly_path, index=False)
print(f"💾 Exported hourly dosage to: {output_hourly_path}")

/var/folders/4m/_fg19tl1739c5j7rvwwqvy3m0000gn/T/ipykernel_31010/2313135606.py:54: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



,Datetime,Flow (MGD),HCl Dosage (mg/L),FeCl₃ Dosage (mg/L)
0,2025-08-13 20:00:00-04:00,0.039,"6,142.390",680.781
1,2025-08-13 21:00:00-04:00,0.058,"4,134.787",458.272
2,2025-08-13 22:00:00-04:00,0.066,"3,616.705",400.851
3,2025-08-13 23:00:00-04:00,0.083,"2,884.142",319.659
4,2025-08-14 00:00:00-04:00,0.113,"2,110.875",233.955
5,2025-08-14 01:00:00-04:00,0.113,"2,106.662",233.488
6,2025-08-14 02:00:00-04:00,0.114,"2,105.614",233.372
7,2025-08-14 03:00:00-04:00,0.118,"2,024.663",224.400
8,2025-08-14 04:00:00-04:00,0.140,"1,703.012",188.750
9,2025-08-14 05:00:00-04:00,0.151,"1,587.342",175.930


💾 Exported hourly dosage to: /Users/mutsamungoshi/Documents/Wastewater Treatment/data/hourly_chemical_dosage.csv


In [41]:
import plotly.graph_objects as go

# --- Hourly Dosage Dual-Axis Plot ---
fig = go.Figure()

# HCl — Primary y-axis (left)
fig.add_trace(go.Scatter(
    x=hourly_flow["Datetime"],
    y=hourly_flow["HCl Dosage (mg/L)"],
    mode="lines",
    name="HCl Dosage (mg/L)",
    line=dict(color="green", width=2),
    yaxis="y1"
))

# FeCl₃ — Secondary y-axis (right)
fig.add_trace(go.Scatter(
    x=hourly_flow["Datetime"],
    y=hourly_flow["FeCl₃ Dosage (mg/L)"],
    mode="lines",
    name="FeCl₃ Dosage (mg/L)",
    line=dict(color="orange", width=2),
    yaxis="y2"
))

# --- Event Overlays ---
fig.add_vrect(
    x0=HCL_OFF_START_DT, x1=HCL_RESUME_DT,
    fillcolor="rgba(0,128,0,0.12)", layer="below", line_width=0,
    annotation_text="HCl OFF", annotation_position="top left",
    annotation=dict(font_color="green")
)

fig.add_vrect(
    x0=FECL3_OFF_DT, x1=FECL3_RESUME_DT,
    fillcolor="rgba(255,140,0,0.12)", layer="below", line_width=0,
    annotation_text="FeCl₃ OFF", annotation_position="top left",
    annotation=dict(font_color="darkorange")
)

fig.update_layout(
    template="plotly_white",
    title="Hourly Chemical Dosage — HCl and FeCl₃ (mg/L)",
    xaxis=dict(
        title="Timestamp",
        showgrid=True,
        rangeslider=dict(visible=True),  # ✅ Enables slider
        type="date"
    ),
    yaxis=dict(
        title="HCl Dosage (mg/L)",
        color="green",
        rangemode="tozero"
    ),
    yaxis2=dict(
        title="FeCl₃ Dosage (mg/L)",
        color="orange",
        overlaying="y",
        side="right",
        rangemode="tozero"
    ),
    legend=dict(
        title="Chemical",
        orientation="h",
        yanchor="bottom",
        y=1.1,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="lightgray",
        borderwidth=1
    ),
    hovermode="x unified",
    margin=dict(t=100, b=60, l=80, r=80),
    updatemenus=[
        dict(
            buttons=list([
                dict(
                    args=[{"yaxis.type": "linear", "yaxis2.type": "linear"}],
                    label="Linear Scale",
                    method="relayout"
                ),
                dict(
                    args=[{"yaxis.type": "log", "yaxis2.type": "log"}],
                    label="Log Scale",
                    method="relayout"
                )
            ]),
            direction="down",
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.2,
            yanchor="top",
            bgcolor="white",
            bordercolor="gray",
            borderwidth=1
        )
    ]
)

# --- Show and Export ---
fig.show()
fig.write_html("Hourly_Chemical_Dosage.html", include_plotlyjs="embed")

In [31]:
import plotly.graph_objects as go

# --- Round to the nearest hour and average readings ---
nh3_hourly = nh3.copy()
nh3_hourly["Datetime"] = nh3_hourly["Datetime"].dt.floor("H")
nh3_hourly = nh3_hourly.groupby("Datetime", as_index=False)["NH3_ppm"].mean()

h2s_hourly = h2s.copy()
h2s_hourly["Datetime"] = h2s_hourly["Datetime"].dt.floor("H")
h2s_hourly = h2s_hourly.groupby("Datetime", as_index=False)["H2S_ppm"].mean()

# --- Hourly Chemical Dosage Plot (Dual Axis) ---
fig_dosage = go.Figure()

# HCl Dosage — Primary y-axis (left)
fig_dosage.add_trace(go.Scatter(
    x=hourly_combined_clean["Datetime"],
    y=hourly_combined_clean["HCl Dosage (mg/L)"],
    mode="lines",
    name="HCl Dosage (mg/L)",
    line=dict(color="green", width=2),
    yaxis="y1"
))

# FeCl₃ Dosage — Secondary y-axis (right)
fig_dosage.add_trace(go.Scatter(
    x=hourly_combined_clean["Datetime"],
    y=hourly_combined_clean["FeCl₃ Dosage (mg/L)"],
    mode="lines",
    name="FeCl₃ Dosage (mg/L)",
    line=dict(color="orange", width=2),
    yaxis="y2"
))

# --- Highlight OFF Windows ---
fig_dosage.add_vrect(
    x0=HCL_OFF_START_DT, x1=HCL_RESUME_DT,
    fillcolor="rgba(0,128,0,0.12)", layer="below", line_width=0,
    annotation_text="HCl OFF", annotation_position="top left",
    annotation=dict(font_color="green")
)

fig_dosage.add_vrect(
    x0=FECL3_OFF_DT, x1=FECL3_RESUME_DT,
    fillcolor="rgba(255,140,0,0.12)", layer="below", line_width=0,
    annotation_text="FeCl₃ OFF", annotation_position="top left",
    annotation=dict(font_color="darkorange")
)

# Optional: Vertical line for sensor removal
add_dt_vline(fig_dosage, SENSORS_REMOVED, text="Meters Removed")

# --- Layout ---
fig_dosage.update_layout(
    template="plotly_white",
    title="Hourly Chemical Dosage — HCl and FeCl₃ (mg/L)",
    xaxis=dict(title="Timestamp", showgrid=True),
    yaxis=dict(
        title="HCl Dosage (mg/L)", 
        color="green", 
        rangemode="tozero"
    ),
    yaxis2=dict(
        title="FeCl₃ Dosage (mg/L)", 
        color="orange", 
        overlaying="y", 
        side="right",
        rangemode="tozero"
    ),
    legend=dict(
        title="Chemical",
        orientation="h",
        yanchor="bottom",
        y=1.1,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="lightgray",
        borderwidth=1
    ),
    hovermode="x unified",
    margin=dict(t=100, b=60, l=80, r=80)
)

fig_dosage.show()

# Save to standalone HTML
fig_dosage.write_html("Hourly_Chemical_Dosage.html", include_plotlyjs="embed")

/var/folders/4m/_fg19tl1739c5j7rvwwqvy3m0000gn/T/ipykernel_31010/3715848534.py:5: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

/var/folders/4m/_fg19tl1739c5j7rvwwqvy3m0000gn/T/ipykernel_31010/3715848534.py:9: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



In [26]:
# --- Plot 2: Gas Levels Only ---
fig_gases = go.Figure()

# NH₃ (Left Axis)
fig_gases.add_trace(go.Scatter(
    x=hourly_combined_clean["Datetime"],
    y=hourly_combined_clean["NH3_ppm"],
    mode="lines",
    name="NH₃ (ppm)",
    line=dict(color="#00AEEF", width=2),
    yaxis="y1"
))

# H₂S (Right Axis)
fig_gases.add_trace(go.Scatter(
    x=hourly_combined_clean["Datetime"],
    y=hourly_combined_clean["H2S_ppm"],
    mode="lines",
    name="H₂S (ppm)",
    line=dict(color="#8B0000", width=2),
    yaxis="y2"
))

# Event Windows
fig_gases.add_vrect(x0=HCL_OFF_START_DT, x1=HCL_RESUME_DT, fillcolor="rgba(0,128,0,0.12)", layer="below", line_width=0,
                    annotation_text="HCl OFF", annotation_position="top left", annotation=dict(font_color="green"))
fig_gases.add_vrect(x0=FECL3_OFF_DT, x1=FECL3_RESUME_DT, fillcolor="rgba(255,140,0,0.12)", layer="below", line_width=0,
                    annotation_text="FeCl₃ OFF", annotation_position="top left", annotation=dict(font_color="darkorange"))
add_dt_vline(fig_gases, SENSORS_REMOVED, text="Meters Removed")

fig_gases.update_layout(
    template="plotly_white",
    title="Hourly Gas Concentrations — NH₃ and H₂S (ppm)",
    xaxis=dict(title="Timestamp", showgrid=True),
    yaxis=dict(title="NH₃ (ppm)", color="#00AEEF", rangemode="tozero"),
    yaxis2=dict(title="H₂S (ppm)", color="#8B0000", overlaying="y", side="right", rangemode="tozero"),
    legend=dict(
        title="Gases",
        orientation="h",
        yanchor="bottom",
        y=1.1,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="lightgray",
        borderwidth=1
    ),
    hovermode="x unified",
    margin=dict(t=100, b=60, l=80, r=80)
)

fig_gases.show()
fig_gases.write_html("Hourly_Gas_Levels.html", include_plotlyjs="embed")

In [22]:
outliers = hourly_combined[hourly_combined["Flow (MGD)"] <= 0.05]
display(outliers[["Datetime", "Flow (MGD)", "HCl Dosage (mg/L)", "FeCl₃ Dosage (mg/L)"]])
outliers = hourly_combined[hourly_combined["Flow (MGD)"] <= 0.05]
outliers.to_csv("hourly_dosage_outliers.csv", index=False)

,Datetime,Flow (MGD),HCl Dosage (mg/L),FeCl₃ Dosage (mg/L)
0,2025-08-13 20:00:00-04:00,0.039,"6,142.390",680.781
24,2025-08-15 20:00:00-04:00,0.026,"9,112.692","1,009.989"
25,2025-08-15 21:00:00-04:00,0.046,"5,198.400",576.155
26,2025-08-15 22:00:00-04:00,0.047,"5,077.415",562.746
48,2025-08-16 20:00:00-04:00,0.015,"16,129.302","1,787.662"
...,...,...,...,...
1259,2025-10-06 07:00:00-04:00,0.044,"5,462.226",605.396
1260,2025-10-06 08:00:00-04:00,0.044,"5,455.687",604.671
1261,2025-10-06 09:00:00-04:00,0.044,"5,448.505",603.875
1262,2025-10-06 10:00:00-04:00,0.044,"5,441.213",603.067


In [33]:
import pandas as pd

# Load the full dataset (with outliers included)
hourly_combined_outliers = pd.read_csv("hourly_dosage_outliers.csv", parse_dates=["Datetime"])

# Check structure (optional)
hourly_combined_outliers.info()


import plotly.graph_objects as go

# --- Round to the nearest hour and average readings ---
nh3_hourly = nh3.copy()
nh3_hourly["Datetime"] = nh3_hourly["Datetime"].dt.floor("H")
nh3_hourly = nh3_hourly.groupby("Datetime", as_index=False)["NH3_ppm"].mean()

h2s_hourly = h2s.copy()
h2s_hourly["Datetime"] = h2s_hourly["Datetime"].dt.floor("H")
h2s_hourly = h2s_hourly.groupby("Datetime", as_index=False)["H2S_ppm"].mean()

# --- Hourly Chemical Dosage Plot (Dual Axis) — With OUTLIERS ---
fig_dosage = go.Figure()

# HCl Dosage — Primary y-axis (left)
fig_dosage.add_trace(go.Scatter(
    x=hourly_combined_outliers["Datetime"],
    y=hourly_combined_outliers["HCl Dosage (mg/L)"],
    mode="lines",
    name="HCl Dosage (mg/L)",
    line=dict(color="green", width=2),
    yaxis="y1"
))

# FeCl₃ Dosage — Secondary y-axis (right)
fig_dosage.add_trace(go.Scatter(
    x=hourly_combined_outliers["Datetime"],
    y=hourly_combined_outliers["FeCl₃ Dosage (mg/L)"],
    mode="lines",
    name="FeCl₃ Dosage (mg/L)",
    line=dict(color="orange", width=2),
    yaxis="y2"
))

# --- Highlight OFF Windows ---
fig_dosage.add_vrect(
    x0=HCL_OFF_START_DT, x1=HCL_RESUME_DT,
    fillcolor="rgba(0,128,0,0.12)", layer="below", line_width=0,
    annotation_text="HCl OFF", annotation_position="top left",
    annotation=dict(font_color="green")
)

fig_dosage.add_vrect(
    x0=FECL3_OFF_DT, x1=FECL3_RESUME_DT,
    fillcolor="rgba(255,140,0,0.12)", layer="below", line_width=0,
    annotation_text="FeCl₃ OFF", annotation_position="top left",
    annotation=dict(font_color="darkorange")
)

# Optional: Vertical line for sensor removal
add_dt_vline(fig_dosage, SENSORS_REMOVED, text="Meters Removed")

# --- Layout ---
fig_dosage.update_layout(
    template="plotly_white",
    title="Hourly Chemical Dosage (With Outliers) — HCl and FeCl₃ (mg/L)",
    xaxis=dict(title="Timestamp", showgrid=True),
    yaxis=dict(
        title="HCl Dosage (mg/L)", 
        color="green", 
        rangemode="tozero"
    ),
    yaxis2=dict(
        title="FeCl₃ Dosage (mg/L)", 
        color="orange", 
        overlaying="y", 
        side="right",
        rangemode="tozero"
    ),
    legend=dict(
        title="Chemical",
        orientation="h",
        yanchor="bottom",
        y=1.1,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="lightgray",
        borderwidth=1
    ),
    hovermode="x unified",
    margin=dict(t=100, b=60, l=80, r=80)
)

fig_dosage.show()

# Save to standalone HTML
fig_dosage.write_html("Hourly_Chemical_Dosage_With_Outliers.html", include_plotlyjs="embed")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype                    
---  ------               --------------  -----                    
 0   Datetime             150 non-null    datetime64[ns, UTC-04:00]
 1   Flow (Gal)           150 non-null    float64                  
 2   Flow (MGD)           150 non-null    float64                  
 3   HCl Dosage (mg/L)    150 non-null    float64                  
 4   FeCl₃ Dosage (mg/L)  150 non-null    float64                  
 5   NH3_ppm              82 non-null     float64                  
 6   H2S_ppm              82 non-null     float64                  
dtypes: datetime64[ns, UTC-04:00](1), float64(6)
memory usage: 8.3 KB


/var/folders/4m/_fg19tl1739c5j7rvwwqvy3m0000gn/T/ipykernel_31010/2646515153.py:14: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

/var/folders/4m/_fg19tl1739c5j7rvwwqvy3m0000gn/T/ipykernel_31010/2646515153.py:18: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



### ⚠️ Expanded Analysis of Low-Flow Dosage Outliers (n = 150)

We analyzed 150 hourly records with **Flow ≤ 0.05 MGD**, which were excluded from final visualizations due to extreme dosage values.

#### 🕓 Temporal Distribution by Hour
- Outliers occurred across a wide range of hours, not just early mornings.
- Clusters were observed during:
  - **Morning window**: 06:00–11:00 AM
  - **Evening window**: 08:00–11:00 PM
- This suggests that low-flow anomalies may be linked to **operational transitions**, **low-demand periods**, or **instrument lag** — not simply time-of-day.

#### 📅 Distribution by Day of Week
- Outliers peaked on **Mondays**, **Fridays**, and especially **Thursdays**.
- **Thursday outliers** had the most extreme FeCl₃ dosage values:
  - Mean ≈ **103,552 mg/L**
  - Max ≈ **539,915 mg/L**
- **Friday** also showed:
  - Max HCl dosage ≈ **16,421 mg/L**
  - Max FeCl₃ dosage ≈ **519,687 mg/L**

These dosage spikes are caused by dividing a constant chemical feed by a near-zero flow:

\[
\text{Dosage (mg/L)} = \frac{\text{lbs}}{8.34 \times \text{MGD}}
\]

As MGD → 0, dosage → ∞. Thus, these rows were excluded using:

```python
hourly_combined_clean = hourly_combined[hourly_combined["Flow (MGD)"] > 0.05]


## 6) Visualizations
### 6.1 NH₃ and H₂S vs time (with OFF windows)


In [18]:
import plotly.graph_objects as go

# ============================================================
# 🧩 NH₃ & H₂S Time-Series Visualization (Dual Y-Axis)
# ============================================================
if not daily.empty and (("NH3_ppm" in daily.columns) or ("H2S_ppm" in daily.columns)):

    time_col = "LocalDate" if "LocalDate" in daily.columns else "Datetime"

    fig = go.Figure()

    # ------------------------------------------------------------
    # NH₃ — Left Axis
    # ------------------------------------------------------------
    if "NH3_ppm" in daily.columns:
        fig.add_trace(go.Scatter(
            x=daily[time_col],
            y=daily["NH3_ppm"],
            mode="lines+markers",
            name="NH₃ (ppm)",
            line=dict(color="#00AEEF", width=2),
            yaxis="y1"
        ))

    # ------------------------------------------------------------
    # H₂S — Right Axis
    # ------------------------------------------------------------
    if "H2S_ppm" in daily.columns:
        fig.add_trace(go.Scatter(
            x=daily[time_col],
            y=daily["H2S_ppm"],
            mode="lines+markers",
            name="H₂S (ppm)",
            line=dict(color="#8B0000", width=2),
            yaxis="y2"
        ))

    # ========================================================
    # Operational Windows and Event Lines
    # ========================================================
    fig.add_vrect(
        x0=HCL_OFF_START, x1=HCL_RESUME,
        fillcolor="rgba(255, 0, 0, 0.12)", layer="below", line_width=0,
        annotation_text="HCl OFF", annotation_position="top left",
        annotation=dict(font_color="red")
    )
    fig.add_vrect(
        x0=FERRIC_OFF, x1=FERRIC_ON,
        fillcolor="rgba(0, 0, 255, 0.12)", layer="below", line_width=0,
        annotation_text="FeCl₃ OFF", annotation_position="top left",
        annotation=dict(font_color="blue")
    )
    add_dt_vline(fig, SENSORS_REMOVED, text="Meters Removed")

    # ========================================================
    # Layout: Dual Axis Styling
    # ========================================================
    fig.update_layout(
        template="plotly_white",
        title="NH₃ and H₂S — Daily Means with Operational Windows (Dual Axis)",
        title_font=dict(size=18),
        xaxis=dict(title="Date"),
        yaxis=dict(
            title="NH₃ Concentration (ppm)",
            color="#00AEEF",
            rangemode="tozero"
        ),
        yaxis2=dict(
            title="H₂S Concentration (ppm)",
            color="#8B0000",
            overlaying="y",
            side="right",
            rangemode="tozero"
        ),
        legend=dict(
            title="Gas Type",
            orientation="h",
            yanchor="bottom", y=-0.25,
            xanchor="center", x=0.5
        ),
        hovermode="x unified",
        margin=dict(t=80, b=60, l=80, r=80)
    )

    fig.show()

else:
    print("⚠️ NH₃/H₂S not available in `daily`.")

fig.write_html(
    "NH3_H2S_v_time.html",
    include_plotlyjs="embed",   # embeds full JS library inside file
    full_html=True              # makes it a complete standalone page
)



### 6.2 Chemical dosage time-series (mg/L)


In [8]:
import plotly.express as px

# ============================================================
# 🧪 Daily Chemical Dosage — HCl & FeCl₃ (mg/L)
# ============================================================

# Safely match both naming styles (plain and subscript ₃)
ycols = [c for c in ["HCl Dosage (mg/L)", "FeCl3 Dosage (mg/L)", "FeCl₃ Dosage (mg/L)"] if c in daily.columns]

if not daily.empty and len(ycols) > 0:
    fig = px.line(
        daily,
        x="LocalDate" if "LocalDate" in daily.columns else "Datetime",
        y=ycols,
        title="Daily Chemical Dosage — HCl & FeCl₃ (mg/L)",
        labels={"value": "Dosage (mg/L)", "variable": "Chemical"},
        color_discrete_map={
            "HCl Dosage (mg/L)": "#FF3333",   # acid = red
            "FeCl3 Dosage (mg/L)": "#3333FF",
            "FeCl₃ Dosage (mg/L)": "#3333FF"  # ensure correct color map for subscript version
        }
    )

    # ------------------------------------------------------------
    # Visual polish and event overlays
    # ------------------------------------------------------------
    fig.update_traces(mode="lines+markers", marker=dict(size=6))

    fig.add_vrect(
        x0=HCL_OFF_START, x1=HCL_RESUME,
        fillcolor="rgba(255, 0, 0, 0.12)", layer="below", line_width=0,
        annotation_text="HCl OFF", annotation_position="top left",
        annotation=dict(font_color="red")
    )

    fig.add_vrect(
        x0=FERRIC_OFF, x1=FERRIC_ON,
        fillcolor="rgba(0, 0, 255, 0.12)", layer="below", line_width=0,
        annotation_text="FeCl₃ OFF", annotation_position="top left",
        annotation=dict(font_color="blue")
    )

    add_dt_vline(fig, SENSORS_REMOVED, text="Meters Removed")

    fig.update_layout(
        template="plotly_white",
        title_font=dict(size=18),
        xaxis_title="Date",
        yaxis_title="Chemical Dosage (mg/L)",
        hovermode="x unified",
        legend=dict(
            title="Chemical Type",
            orientation="h",
            yanchor="bottom", y=-0.35,
            xanchor="center", x=0.5
        ),
        margin=dict(t=80, b=60, l=80, r=60)
    )

    fig.show()

else:
    print("⚠️ Dosage columns not present.")

fig.write_html(
    "daily_dosage.html",
    include_plotlyjs="embed",   # embeds full JS library inside file
    full_html=True              # makes it a complete standalone page
)


### 6.3 NH₃ vs HCl dosage and H₂S vs FeCl₃ dosage (scatter)


In [9]:
import plotly.express as px

# ============================================================
# 🧪 Correlation Plots: Gas Concentration vs Chemical Dosage
# ============================================================

pairs = [
    ("NH3_ppm", "HCl Dosage (mg/L)", "NH₃ vs HCl Dosage"),
    ("H2S_ppm", "FeCl3 Dosage (mg/L)", "H₂S vs FeCl₃ Dosage"),
    ("H2S_ppm", "FeCl₃ Dosage (mg/L)", "H₂S vs FeCl₃ Dosage (subscript)")  # fallback for proper Unicode
]

for dep, indep, title in pairs:
    # Handle both FeCl3 naming variants
    possible_matches = [c for c in daily.columns if c.strip().startswith(indep.split()[0])]
    if dep in daily.columns and possible_matches:
        indep_col = possible_matches[0]

        fig = px.scatter(
            daily,
            x=indep_col,
            y=dep,
            trendline="ols",
            title=title,
            labels={
                indep_col: f"{indep_col} (mg/L)",
                dep: f"{dep} (ppm)"
            },
            opacity=0.8
        )

        # ------------------------------------------------------------
        # Style polish
        # ------------------------------------------------------------
        fig.update_traces(marker=dict(size=8, color="#4444AA", line=dict(width=0.5, color="white")))
        fig.update_layout(
            template="plotly_white",
            title_font=dict(size=18),
            xaxis_title=indep_col,
            yaxis_title=dep,
            hovermode="closest",
            margin=dict(t=80, b=60, l=80, r=60),
            showlegend=False
        )

        # Make the trendline visible (especially with sparse data)
        fig.data[1].line.color = "#FF0000"
        fig.data[1].line.width = 3
        fig.data[1].name = "OLS Trendline"

        fig.show()

        fig.write_html(
        "dosages_scatter.html",
        include_plotlyjs="embed",   # embeds full JS library inside file
        full_html=True              # makes it a complete standalone page
)



## 7) Before/After analysis (ON vs OFF)
- Compare daily means for NH₃/H₂S in **OFF windows** vs **outside windows**.


In [10]:
# ============================================================
# 🧮 Compare mean gas concentrations during ON/OFF periods
# ============================================================
import numpy as np
import pandas as pd
import pytz

# --- make sure LocalDate is parsed as datetime (Eastern) ---
daily["LocalDate"] = pd.to_datetime(daily["LocalDate"], errors="coerce")
daily["_ET"] = daily["LocalDate"].dt.tz_localize("US/Eastern")

# --- helper to flag within a window ---
def flag_window(ts, start, end):
    """Return boolean Series for timestamps within [start, end]."""
    return (ts >= start) & (ts <= end)

# --- operational windows ---
def to_eastern(ts):
    t = pd.to_datetime(ts)
    if t.tzinfo is None:
        return t.tz_localize("US/Eastern")
    return t.tz_convert("US/Eastern")

HCL_OFF_START  = to_eastern("2025-08-27 00:00")
HCL_RESUME     = to_eastern("2025-10-02 14:30")
FERRIC_OFF     = to_eastern("2025-09-09 08:00")
FERRIC_ON      = to_eastern("2025-09-17 12:00")

# --- flag OFF periods ---
daily["HCl_OFF"]   = flag_window(daily["_ET"], HCL_OFF_START, HCL_RESUME).astype(int)
daily["FeCl3_OFF"] = flag_window(daily["_ET"], FERRIC_OFF, FERRIC_ON).astype(int)

# --- compute ON/OFF mean concentrations ---
summary = {}
for gas in ["NH3_ppm", "H2S_ppm"]:
    if gas in daily.columns:
        hcl_off = daily.loc[daily["HCl_OFF"] == 1, gas].mean()
        hcl_on  = daily.loc[daily["HCl_OFF"] == 0, gas].mean()
        fe_off  = daily.loc[daily["FeCl3_OFF"] == 1, gas].mean()
        fe_on   = daily.loc[daily["FeCl3_OFF"] == 0, gas].mean()

        summary[gas] = {
            "HCl_OFF_mean": hcl_off,
            "HCl_ON_mean":  hcl_on,
            "ΔHCl_%": 100 * (hcl_off - hcl_on) / hcl_on if pd.notna(hcl_on) and hcl_on != 0 else np.nan,
            "Fe_OFF_mean": fe_off,
            "Fe_ON_mean":  fe_on,
            "ΔFe_%": 100 * (fe_off - fe_on) / fe_on if pd.notna(fe_on) and fe_on != 0 else np.nan,
        }

result = pd.DataFrame(summary).T.round(3)
display(
    result.style.format({
        "HCl_OFF_mean": "{:.3f}",
        "HCl_ON_mean": "{:.3f}",
        "ΔHCl_%": "{:+.1f}%",
        "Fe_OFF_mean": "{:.3f}",
        "Fe_ON_mean": "{:.3f}",
        "ΔFe_%": "{:+.1f}%"
    }).background_gradient(cmap="coolwarm", axis=None)
)

# --- clean helper column ---
daily.drop(columns=["_ET"], inplace=True, errors="ignore")

,HCl_OFF_mean,HCl_ON_mean,ΔHCl_%,Fe_OFF_mean,Fe_ON_mean,ΔFe_%
NH3_ppm,20.142,9.388,+114.6%,19.183,17.741,+8.1%
H2S_ppm,0.010,0.360,-97.3%,0.001,0.100,-99.1%



## 8) Correlations


In [11]:
import pandas as pd
import numpy as np
import plotly.express as px

# ------------------------------------------------------------
# Correlation matrix for gas, flow, and chemical dosage
# ------------------------------------------------------------

corr_cols = [
    c for c in [
        "NH3_ppm",
        "H2S_ppm",
        "Flow (MGD)",
        "HCl Dosage (mg/L)",
        "FeCl₃ Dosage (mg/L)",
        "FeCl3 Dosage (mg/L)"  # fallback if ASCII name
    ] if c in daily.columns
]

if len(corr_cols) >= 3:
    # Compute correlation
    corr = daily[corr_cols].corr(method="pearson").round(2)

    # Melt for Plotly Express
    corr_melt = corr.reset_index().melt(id_vars="index")
    corr_melt.columns = ["Variable 1", "Variable 2", "Correlation"]

    # Plot heatmap
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale="RdBu_r",
        zmin=-1, zmax=1,
        aspect="auto",
        title="Daily Correlation Matrix — NH₃, H₂S, Flow, and Dosages",
    )

    fig.update_traces(
        texttemplate="%{z:.2f}",
        textfont_size=12
    )

    fig.update_layout(
        template="plotly_white",
        xaxis_title="",
        yaxis_title="",
        coloraxis_colorbar=dict(
            title="Correlation",
            tickvals=[-1, -0.5, 0, 0.5, 1],
            ticktext=["−1", "−0.5", "0", "+0.5", "+1"]
        ),
        title_font=dict(size=18),
        margin=dict(t=80, b=40, l=60, r=40)
    )

    fig.show()

else:
    print("⚠️ Not enough columns for correlations.", corr_cols)

fig.write_html(
    "correlation_matrix.html",
    include_plotlyjs="embed",   # embeds full JS library inside file
    full_html=True              # makes it a complete standalone page
)

In [12]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ============================================================
# 🔁 Lagged Correlation Analysis — FeCl₃ Dosage vs H₂S
# ============================================================

# --- Handle both naming variants (FeCl3 and FeCl₃) ---
ferric_col = next((c for c in daily.columns if "FeCl" in c and "Dosage" in c), None)
if not ferric_col or "H2S_ppm" not in daily.columns:
    print("⚠️ Missing required columns for lag analysis.")
else:
    # --- Optional: filter to post-resumption phase ---
    mask = daily["LocalDate"] >= pd.Timestamp("2025-09-17")   # after FeCl₃ ON
    subset = daily.loc[mask].copy()

    # --- Smooth H₂S to reduce daily noise ---
    subset["H2S_smooth"] = subset["H2S_ppm"].rolling(3, min_periods=1).mean()

    lags = range(0, 15)  # test up to 14-day lag
    corrs = []

    for lag in lags:
        shifted = subset.copy()
        shifted[f"{ferric_col}_lag{lag}"] = shifted[ferric_col].shift(lag)
        r = shifted["H2S_smooth"].corr(shifted[f"{ferric_col}_lag{lag}"])
        corrs.append(dict(Lag_Days=lag, Correlation=r))

    corr_df = pd.DataFrame(corrs).dropna()

    # --- Plot lag correlation bar chart ---
    fig = px.bar(
        corr_df, x="Lag_Days", y="Correlation",
        text="Correlation", color="Correlation",
        color_continuous_scale="RdBu_r", range_color=(-1, 1),
        title="Lagged Correlation — FeCl₃ Dosage vs H₂S Concentration (post-resumption, 0–14 days)"
    )

    fig.update_traces(
        texttemplate=["" if pd.isna(v) else f"{v:.2f}" for v in corr_df["Correlation"]],
        textposition="outside",
        marker_line_width=0.5, marker_line_color="white"
    )

    fig.add_hline(y=0, line_width=1.2, line_dash="dot", line_color="gray")

    fig.update_layout(
        template="plotly_white",
        title_font=dict(size=18),
        xaxis_title="Lag (days after FeCl₃ dosage)",
        yaxis_title="Pearson correlation (r)",
        yaxis_range=[-1, 1],
        coloraxis_colorbar=dict(
            title="Correlation (r)",
            tickvals=[-1, -0.5, 0, 0.5, 1],
            ticktext=["−1", "−0.5", "0", "+0.5", "+1"]
        ),
        margin=dict(t=80, b=60, l=80, r=60)
    )

    fig.show()

    fig.write_html(
    "lagged_correlation_bar.html",
    include_plotlyjs="embed",   # embeds full JS library inside file
    full_html=True              # makes it a complete standalone page
    )

    # ========================================================
    # 🔍 Optional Overlay: Verify the strongest lag visually
    # ========================================================
    best_lag = corr_df.loc[corr_df["Correlation"].abs().idxmax(), "Lag_Days"]
    overlay = subset.copy()
    overlay["FeCl₃_shift"] = overlay[ferric_col].shift(int(best_lag))

    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=overlay["LocalDate"], y=overlay["H2S_smooth"],
        mode="lines+markers", name="H₂S (ppm, smoothed)",
        line=dict(color="firebrick", width=2)
    ))
    fig2.add_trace(go.Scatter(
        x=overlay["LocalDate"], y=overlay["FeCl₃_shift"],
        mode="lines+markers", name=f"FeCl₃ Dosage (shifted {int(best_lag)} days)",
        yaxis="y2", line=dict(color="royalblue", dash="dot", width=2)
    ))

    fig2.update_layout(
        template="plotly_white",
        title=f"H₂S vs FeCl₃ Dosage (Shifted {int(best_lag)} Days — r={corr_df['Correlation'].iloc[best_lag]:.2f})",
        xaxis_title="Date",
        yaxis_title="H₂S (ppm, smoothed)",
        yaxis2=dict(title="FeCl₃ Dosage (mg/L)", overlaying="y", side="right"),
        hovermode="x unified",
        legend=dict(
            orientation="h", yanchor="bottom", y=-0.45,
            xanchor="center", x=0.5
        ),
        margin=dict(t=80, b=60, l=80, r=80)
    )

    fig2.show()

    fig2.write_html(
    "lagged_correlation_10day.html",
    include_plotlyjs="embed",   # embeds full JS library inside file
    full_html=True              # makes it a complete standalone page
    )

In [13]:
import pandas as pd
import plotly.express as px
from scipy.stats import ttest_ind

# ============================================================
# 📦 Phase-Comparison Boxplots — NH₃ & H₂S with Statistical Tests
# ============================================================

# --- Ensure proper categorical ordering for consistent display ---
phase_order = ["Pre-shutdown", "Chemical OFF (HCl & partial Ferric)", "Post-resumption"]
daily["Phase"] = pd.Categorical(daily["Phase"], categories=phase_order, ordered=True)

# --- Select available gas columns ---
gas_cols = [c for c in ["NH3_ppm", "H2S_ppm"] if c in daily.columns]

if not daily.empty and "Phase" in daily.columns and gas_cols:

    print("📊 Phase sample counts:")
    print(daily["Phase"].value_counts(), "\n")

    for gas in gas_cols:
        print(f"🔬 Statistical comparisons for {gas}:")

        # --- Compare mean concentrations between each pair of phases ---
        results = []
        for i in range(len(phase_order)):
            for j in range(i + 1, len(phase_order)):
                ph1, ph2 = phase_order[i], phase_order[j]
                g1 = daily.loc[daily["Phase"] == ph1, gas].dropna()
                g2 = daily.loc[daily["Phase"] == ph2, gas].dropna()
                if len(g1) > 1 and len(g2) > 1:
                    t, p = ttest_ind(g1, g2, equal_var=False)
                    results.append((f"{ph1} vs {ph2}", t, p))
        if results:
            for label, t, p in results:
                sig = "✅ significant" if p < 0.05 else "ns"
                print(f"  {label}: t={t:.2f}, p={p:.4f} ({sig})")
        else:
            print("  ⚠️ Not enough data for statistical comparison.")

        # --- Generate phase boxplot with overlayed data points ---
        fig = px.box(
            daily,
            x="Phase",
            y=gas,
            color="Phase",
            title=f"{gas.replace('_', '').replace('ppm',' (ppm)')} by Operational Phase",
            points="all",  # show individual samples
            color_discrete_sequence=["#4C78A8", "#F58518", "#54A24B"]
        )

        # --- Add mean ± SD markers ---
        fig.update_traces(boxmean="sd")

        # --- Layout polish ---
        fig.update_layout(
            template="plotly_white",
            title_font=dict(size=18),
            xaxis_title="Operational Phase",
            yaxis_title=f"{gas.split('_')[0]} Concentration (ppm)",
            showlegend=False,
            margin=dict(t=80, b=60, l=80, r=60)
        )

        # --- Apply log scale if dynamic range >10× ---
        if daily[gas].max() > 10 * daily[gas].median():
            fig.update_yaxes(
                type="log",
                title=f"{gas.split('_')[0]} Concentration (ppm, log scale)"
            )

        fig.show()
else:
    print("⚠️ Missing Phase or gas concentration columns for boxplot visualization.")

    fig.write_html(
    "box_plot.html",
    include_plotlyjs="embed",   # embeds full JS library inside file
    full_html=True              # makes it a complete standalone page
    )

📊 Phase sample counts:
Phase
Chemical OFF (HCl & partial Ferric)    36
Pre-shutdown                           14
Post-resumption                         6
Name: count, dtype: int64 

🔬 Statistical comparisons for NH3_ppm:
  Chemical OFF (HCl & partial Ferric) vs Post-resumption: t=8.32, p=0.0001 (✅ significant)


🔬 Statistical comparisons for H2S_ppm:
  Chemical OFF (HCl & partial Ferric) vs Post-resumption: t=-3.49, p=0.0175 (✅ significant)


In [14]:
import pandas as pd
import plotly.graph_objects as go

# ============================================================
# ⚗️ Dosage Efficiency Ratios — NH₃/HCl and H₂S/FeCl₃
# ============================================================

# --- Identify correct dosage columns automatically ---
hcl_col = next((c for c in daily.columns if "HCl" in c and "Dosage" in c), None)
fecl_col = next((c for c in daily.columns if "FeCl" in c and "Dosage" in c), None)

# --- Compute ratios safely ---
daily["HCl_Efficiency"] = daily["NH3_ppm"] / daily[hcl_col] if hcl_col in daily.columns else None
daily["FeCl3_Efficiency"] = daily["H2S_ppm"] / daily[fecl_col] if fecl_col in daily.columns else None

# --- Apply smoothing (3-day rolling average) ---
daily["HCl_Efficiency_smooth"] = daily["HCl_Efficiency"].rolling(window=3, min_periods=1).mean()
daily["FeCl3_Efficiency_smooth"] = daily["FeCl3_Efficiency"].rolling(window=3, min_periods=1).mean()

# ============================================================
# 📊 Build Interactive Plot
# ============================================================
fig = go.Figure()

# --- NH₃/HCl efficiency line ---
if "HCl_Efficiency_smooth" in daily.columns:
    fig.add_trace(go.Scatter(
        x=daily["LocalDate"], y=daily["HCl_Efficiency_smooth"],
        mode="lines+markers", name="NH₃/HCl Efficiency",
        line=dict(color="#1f77b4", width=3),
        marker=dict(size=6, line=dict(width=0.5, color="white")),
        hovertemplate="Date: %{x}<br>NH₃/HCl: %{y:.4f}"
    ))

# --- H₂S/FeCl₃ efficiency line ---
if "FeCl3_Efficiency_smooth" in daily.columns:
    fig.add_trace(go.Scatter(
        x=daily["LocalDate"], y=daily["FeCl3_Efficiency_smooth"],
        mode="lines+markers", name="H₂S/FeCl₃ Efficiency",
        line=dict(color="#2ca02c", width=3, dash="dot"),
        marker=dict(size=6, line=dict(width=0.5, color="white")),
        hovertemplate="Date: %{x}<br>H₂S/FeCl₃: %{y:.4f}"
    ))

# ============================================================
# 🧱 Phase shading
# ============================================================
if "Phase" in daily.columns:
    phase_colors = {
        "Pre-shutdown": "#E5F5E0",
        "Chemical OFF (HCl & partial Ferric)": "#FEE0B6",
        "Post-resumption": "#DEEBF7"
    }
    for phase, df_sub in daily.groupby("Phase"):
        if len(df_sub) > 0:
            start, end = df_sub["LocalDate"].min(), df_sub["LocalDate"].max()
            color = phase_colors.get(phase, "lightgray")
            fig.add_vrect(
                x0=start, x1=end,
                fillcolor=color, opacity=0.25, line_width=0,
                annotation_text=phase, annotation_position="top left",
                annotation_font=dict(size=12, color="#444")
            )

# ============================================================
# 🎨 Layout polish
# ============================================================
fig.add_hline(
    y=0.05, line_color="gray", line_dash="dot",
    annotation_text="Lower ratio = better efficiency",
    annotation_position="bottom right", annotation_font=dict(size=12, color="gray")
)

fig.update_yaxes(
    type="log",
    tickvals=[0.001, 0.01, 0.1, 1],
    ticktext=["0.001", "0.01", "0.1", "1"],
    title="Gas (ppm) per mg/L chemical (log scale)"
)

fig.update_layout(
    title=dict(
        text="Dosage Efficiency Indicator — NH₃/HCl & H₂S/FeCl₃ Ratios (Smoothed)",
        font=dict(size=20, color="#333"),
        x=0.5, xanchor="center"
    ),
    template="plotly_white",
    xaxis_title="Date",
    legend=dict(
        orientation="h",
        yanchor="bottom", y=-0.3,
        xanchor="center", x=0.5,
        bgcolor="rgba(255,255,255,0.6)"
    ),
    margin=dict(t=90, b=70, l=80, r=60),
    plot_bgcolor="white"
)

fig.show()
fig.write_html(
    "dosage_efficiency.html",
    include_plotlyjs="embed",   # embeds full JS library inside file
    full_html=True              # makes it a complete standalone page
    )

/var/folders/4m/_fg19tl1739c5j7rvwwqvy3m0000gn/T/ipykernel_31010/1841206173.py:54: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

